<a href="https://colab.research.google.com/github/ahalsey2323405/COSC-650---Applied-LLM-Systems/blob/Week4/week4_tool_use_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 (starter): Multi-Tool Assistant

Everything below runs with no API key. The three tools, the validator, the dispatcher, and the dispatch loop are a **worked example** in a toy domain (arithmetic and unit conversion), driven by a scripted list of tool calls. They are the reference, not your submission.

Build your assistant in a domain you choose. The **TODO (you)** comments cover Parts 1 to 4; Part 5 is the submission checklist. The example uses only the Python standard library. For live calls through the OpenAI-compatible endpoint, install the client with `pip install openai`. Setting a key alone does not enable live calls.

In [1]:
# Gemini import
from openai import OpenAI
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [2]:
import os, ast, operator, json, math, time, multiprocessing

HAS_API_KEY = bool(os.environ.get('GEMINI_API_KEY', '').strip())
print('GEMINI_API_KEY set:', HAS_API_KEY)
# The scripted loop below runs either way. Wiring the live model call is yours (Part 1).

GEMINI_API_KEY set: False


#Part 1: Tool definition


In [3]:
TOOLS = [
    {
        "name": "find_rides","description": "Find available rides between two locations.","parameters": {
            "type": "object","properties": {"pickup": {"type": "string"},"destination": {"type": "string"},
                "ride_type": {"type": "string","enum": ["economy", "premium", "xl"]}},
            "required": ["pickup", "destination", "ride_type"],"additionalProperties": False}},
    {
        "name": "estimate_fare","description": "Estimate the fare for a ride.","parameters": {
            "type": "object","properties": {"distance_miles": {"type": "number"},"ride_type": {
                    "type": "string","enum": ["economy", "premium", "xl"]}},
            "required": ["distance_miles", "ride_type"],"additionalProperties": False}},
    {
        "name": "run_python","description": "Calculate a ride cost using a guarded arithmetic expression.","parameters": {
            "type": "object","properties": {"expression": {"type": "string"}},
            "required": ["expression"],"additionalProperties": False
    }
}
]
print('tools:', [t['name'] for t in TOOLS])

tools: ['find_rides', 'estimate_fare', 'run_python']


#Part 2: Function Definition

In [4]:
class ToolArgError(Exception):
    pass

def validate(name, args):
    spec = next((t['parameters'] for t in TOOLS if t['name'] == name), None)
    if spec is None:
        raise ToolArgError(f'unknown tool: {name}')
    if not isinstance(args, dict):
        raise ToolArgError('arguments must be a JSON object')
    for r in spec.get('required', []):
        if r not in args:
            raise ToolArgError(f'missing required field: {r}')
    for k, v in args.items():
        p = spec['properties'].get(k)
        if p is None:
            raise ToolArgError(f'unexpected field: {k}')
        if p['type'] == 'string':
            if not isinstance(v, str):
                raise ToolArgError(f'{k} must be a string')
        elif p['type'] == 'number':
            if type(v) not in (int, float):
                raise ToolArgError(f'{k} must be a number')
            if isinstance(v, float) and not math.isfinite(v):
                raise ToolArgError(f'{k} must be finite')
        if 'enum' in p and v not in p['enum']:
            raise ToolArgError(f'{k}={v!r} not in {p["enum"]}')

def find_ride(pickup, destination, ride_type):
    return {
        'driver': 'Driver_101',
        'ride_type': ride_type,
        'pickup': pickup,
        'destination': destination,
        'estimated_distance_miles': 8
    }

def estimate_fare(distance_miles, ride_type):
    rates = {'economy': 2.00, 'premium': 3.50, 'xl': 4.50}
    return round(distance_miles * rates[ride_type], 2)

_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg
}

def _safe(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe(node.left), _safe(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe(node.operand))
    raise ValueError('only basic arithmetic operations are allowed')

def _run_expression(expression, result):
    try:
        result.put(_safe(ast.parse(expression, mode='eval').body))
    except Exception as e:
        result.put(e)


# Code-runner guardrail: The run_python tool uses an AST allowlist that permits only numeric constants and basic arithmetic operations: addition, subtraction, multiplication,
# division, exponentiation, and unary negation. It blocks filesystem access, network access, process execution, imports, function calls, attribute access, variables, and
# other Python constructs. Expressions are limited to 100 characters, exponentiation is limited to prevent excessive computation, and execution is terminated after one second.

def run_python(expression):
    blocked = ['import', 'open(', 'os.', 'sys.', 'subprocess', 'socket', '__', 'exec(', 'eval(']
    if any(word in expression for word in blocked):
        raise ValueError('blocked operation: filesystem, network, process execution, imports, dynamic execution')
    if len(expression) > 100:
        raise ValueError('expression is too long')

    result = multiprocessing.Queue()
    process = multiprocessing.Process(target=_run_expression, args=(expression, result))
    process.start()
    process.join(1)

    if process.is_alive():
        process.terminate()
        raise TimeoutError('execution exceeded 1 second')

    output = result.get()
    if isinstance(output, Exception):
        raise output
    return output

IMPL = {
    'find_rides': find_ride,
    'estimate_fare': estimate_fare,
    'run_python': run_python
}

def dispatch(name, args):
    try:
        validate(name, args)
        output = IMPL[name](**args)
        json.dumps(output, allow_nan=False)
        return {'ok': True, 'tool': name, 'output': output}
    except ToolArgError as e:
        return {'ok': False, 'tool': name, 'error_type': 'invalid_arguments', 'message': str(e)}
    except Exception as e:
        return {'ok': False, 'tool': name, 'error_type': 'execution_error', 'message': str(e)}



#Part 3: Testing the tools with prompts

In [5]:
def run_agent(query):
    messages = [
        {"role": "system", "content": "You are a ride-booking assistant. Use the available tools to help users find rides and estimate fares."},
        {"role": "user", "content": query}
    ]

    while True:
        resp = client.chat.completions.create(
            model="gemini-3.6-flash",
            messages=messages,
            tools=[
                {"type": "function", "function": {
                    "name": t["name"],
                    "description": t["description"],
                    "parameters": t["parameters"]
                }}
                for t in TOOLS
            ]
        )

        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print("FINAL:", msg.content)
            return

        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            result = dispatch(call.function.name, args)
            print("TOOL:", call.function.name)
            print("ARGS:", args)
            print("RESULT:", result)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result)
            })




InternalServerError: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]

In [10]:
run_agent("Find me an premium ride from Campus to Downtown.")

TOOL: find_rides
ARGS: {'pickup': 'Campus', 'destination': 'Downtown', 'ride_type': 'premium'}
RESULT: {'ok': True, 'tool': 'find_rides', 'output': {'driver': 'Driver_101', 'ride_type': 'premium', 'pickup': 'Campus', 'destination': 'Downtown', 'estimated_distance_miles': 8}}
TOOL: estimate_fare
ARGS: {'ride_type': 'premium', 'distance_miles': 8}
RESULT: {'ok': True, 'tool': 'estimate_fare', 'output': 28.0}
FINAL: I found a premium ride for you from **Campus** to **Downtown**:

* **Driver:** Driver_101
* **Ride Type:** Premium
* **Estimated Distance:** 8 miles
* **Estimated Fare:** $28.00


In [6]:
run_agent("Estimate the fare for a 12 mile premium ride.")


TOOL: estimate_fare
ARGS: {'distance_miles': 12, 'ride_type': 'premium'}
RESULT: {'ok': True, 'tool': 'estimate_fare', 'output': 42.0}
FINAL: The estimated fare for a 12-mile premium ride is $42.00.


In [7]:
run_agent("Find an economy ride from Campus to Downtown. Then use the estimated distance returned by that tool to estimate the fare.")

TOOL: find_rides
ARGS: {'pickup': 'Campus', 'ride_type': 'economy', 'destination': 'Downtown'}
RESULT: {'ok': True, 'tool': 'find_rides', 'output': {'driver': 'Driver_101', 'ride_type': 'economy', 'pickup': 'Campus', 'destination': 'Downtown', 'estimated_distance_miles': 8}}
TOOL: estimate_fare
ARGS: {'distance_miles': 8, 'ride_type': 'economy'}
RESULT: {'ok': True, 'tool': 'estimate_fare', 'output': 16.0}
FINAL: An economy ride from Campus to Downtown is available with Driver_101, with an estimated distance of 8 miles. 

Based on an estimated distance of 8 miles, the estimated fare for an economy ride is **$16.00**.


In [11]:
run_agent("Use the Python tool to calculate the cost of an economy ride that is 8 miles at $2 per mile.")

TOOL: run_python
ARGS: {'expression': '8 * 2'}
RESULT: {'ok': True, 'tool': 'run_python', 'output': 16}


RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 33.842055165s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'}, '

#Part 4: Failure and recovery

In [8]:
run_agent("Can you find me a ride from Campus to Downtown using a helicopter?")

# Given an incorrect value for a vehicle type (Helicopter) the prompt does not proceed as this is not an accepted enum value for vehicle_type
# It then attempts to recover the response by asking for followup prompt listing the real available options.

FINAL: I'm sorry, but helicopter rides are not available. The available ride types are **economy**, **premium**, and **xl**. 

Would you like me to look for a ride from Campus to Downtown using one of these options?


In [9]:
#Recovering based on what the above response provided, and supplying a new, working prompt:
run_agent("Can you find me an economy ride from Campus to Downtown?")

TOOL: find_rides
ARGS: {'ride_type': 'economy', 'pickup': 'Campus', 'destination': 'Downtown'}
RESULT: {'ok': True, 'tool': 'find_rides', 'output': {'driver': 'Driver_101', 'ride_type': 'economy', 'pickup': 'Campus', 'destination': 'Downtown', 'estimated_distance_miles': 8}}
TOOL: estimate_fare
ARGS: {'distance_miles': 8, 'ride_type': 'economy'}
RESULT: {'ok': True, 'tool': 'estimate_fare', 'output': 16.0}
FINAL: I found an economy ride for you from Campus to Downtown:

* **Driver:** Driver_101
* **Ride Type:** Economy
* **Estimated Distance:** 8 miles
* **Estimated Fare:** $16.00


In [14]:
#Also including this incase you wanted a direct tool call failure of the above failure and not just a demonstration of the model failing and deploying countermeasures

bad_call = {
    "distance_miles": 8,
    "ride_type": "helicopter"
}

failure_result = dispatch("estimate_fare", bad_call)

print("BAD CALL:")
print("TOOL: estimate_fare")
print("ARGS:", bad_call)
print("RESULT:", failure_result)


recovery_call = {
    "distance_miles": 8,
    "ride_type": "premium"
}

recovery_result = dispatch("estimate_fare", recovery_call)

print("\nRECOVERY CALL:")
print("TOOL: estimate_fare")
print("ARGS:", recovery_call)
print("RESULT:", recovery_result)



BAD CALL:
TOOL: estimate_fare
ARGS: {'distance_miles': 8, 'ride_type': 'helicopter'}
RESULT: {'ok': False, 'tool': 'estimate_fare', 'error_type': 'invalid_arguments', 'message': "ride_type='helicopter' not in ['economy', 'premium', 'xl']"}

RECOVERY CALL:
TOOL: estimate_fare
ARGS: {'distance_miles': 8, 'ride_type': 'premium'}
RESULT: {'ok': True, 'tool': 'estimate_fare', 'output': 28.0}


## Parts 1, 3, and 4: dispatch, evaluation, and recovery
The list below sends an invalid unit, then retries the same conversion with a valid unit. Both calls are scripted. No model chooses these calls or reads the errors, even when a key is set. Use this dispatch example to build your model loop, then evaluate your own tools and document a failure with recovery.

## Part 5: Submit
Open a pull request with your schema design write-up, a link to your notebook, and a link to an issue documenting the failure and recovery. Describe your code-runner's allowlist, time limit, and blocked operations. Rubric: schemas (20), loop including a two-step sequence (25), guarded code-runner (20), failure with recovery (20), PR hygiene (15).